### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2025
quarter = 4
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:02 10:40:07


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2025 AND quarter <= 4) 
OR (year = 2025-1 AND quarter >= 4+1)
ORDER BY year DESC, quarter DESC



In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(141, 5)

In [6]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s-1) 
OR (year = %s-1 AND quarter >= %s) 
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2025 AND quarter <= 4-1) 
OR (year = 2025-1 AND quarter >= 4) 
ORDER BY year DESC, quarter DESC


In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8099,10,"7,719,299",4
1,ACE,8099,10,"839,069",4
2,ADVANC,8099,10,"42,863,183",4
3,AEONTS,8099,10,"2,907,113",4
4,AH,8099,10,"745,761",4


In [8]:
dfp.name.unique().shape

(221,)

In [9]:
sql = """
SELECT *
FROM stocks
"""
stocks = pd.read_sql(sql, conlt)
stocks.shape

(226, 15)

In [10]:
sql = """
SELECT *
FROM tickers
"""
tickers = pd.read_sql(sql, conlt)
tickers.shape

(394, 9)

In [11]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
2,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
3,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%
4,AOT,2025,Q4,"18,125,205","18,534,412","-409,207",-2.21%


In [12]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM qt_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 140


In [13]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
tickers.sample(5)

,name,id
257,SABINA,413
10,AIMIRT,669
103,EKH,159
321,TACC,498
330,THCOM,523


In [14]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
df_ins.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
2,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%,9
3,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%,669
4,AOT,2025,Q4,"18,125,205","18,534,412","-409,207",-2.21%,24


In [15]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [17]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
2,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
3,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%
4,AOT,2025,Q4,"18,125,205","18,534,412","-409,207",-2.21%


In [18]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
2,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
3,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%
4,AOT,2025,Q4,"18,125,205","18,534,412","-409,207",-2.21%


In [19]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
7,ASK,2025,Q4,"531,545","388,249","143,296",36.91%
9,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%
13,BANPU,2025,Q4,"-2,024,778","-2,714,635","689,857",25.41%


In [20]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[df_ins_criteria, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
9,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%
16,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%
25,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%


In [21]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
39,DIF,2025,Q4,"13,855,643","868,268","12,987,375",1495.78%,140
16,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
109,SPRC,2025,Q4,"2,569,354","1,641,889","927,465",56.49%,470
25,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74
35,CPNREIT,2025,Q4,"3,460,976","2,333,575","1,127,401",48.31%,647


In [22]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
9,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%,728
16,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
25,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74


In [23]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
1,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
9,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%,728
16,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
25,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74


In [24]:
conlt.commit()
conlt.close()

In [25]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:02 10:40:08
